In [479]:
!python -m pip install --upgrade pip
!python -m pip install dtaidistance
!python -m pip install pandas
!python -m pip install numpy
!python -m pip install matplotlib
!python -m pip install whisper-timestamped
!python -m pip install flask
!python -m pip install audio-similarity


In [480]:
!python -m pip install flask_cors
!python -m pip install flask_restful
!python -m pip install pydub

In [481]:
!python -m pip install librosa
!python -m pip install resampy
!python -m pip install pytube

In [482]:

!python -m pip install praat-parselmouth
!python -m pip install scipy
!python -m pip install python_speech_features

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import whisper_timestamped as whisper
from flask import Flask, send_from_directory, url_for, request
from werkzeug.utils import secure_filename
from flask_cors import CORS #comment this on deployment
from flask_restful import reqparse
from flask import jsonify
import pandas as pd
import io
from ast import literal_eval
import numpy as np
import heapq
import json
import math
import ssl

from pydub import AudioSegment
import librosa
from scipy.io import wavfile
import resampy
from scipy import interpolate
from scipy.fftpack import dct
#import pysptk
from pytube import YouTube
import os
import whisper_timestamped as whisper
from audio_similarity import AudioSimilarity
from dtaidistance import dtw
from dtaidistance import dtw_visualisation as dtwvis
from dtaidistance import dtw_ndim
import parselmouth


In [56]:
TEDTALKBASE="AudioFiles\TedTalkExcerpt.wav"
TEDTALKNOISE="AudioFiles\TedTalkExcerpt_with_noise.wav"
TEDTALKDELAYED="AudioFiles\TedTalkExcerptDelayed.wav"
TEDTALKBASE2="AudioFiles\TedTalkExcerpt2.wav"
TEDTALKNOISE2="AudioFiles\TedTalkExcerpt2_with_noise.wav"
TEDTALKDELAYED2="AudioFiles\TedTalkExcerpt2Delayed.wav"
SONNET1="AudioFiles\Sonnet66Segment1.wav" #Same audio, same phrase, section 1
SONNET2="AudioFiles\Sonnet66Segment2.wav" #Same audio, same phrase, section 2
OZYMANDIAS1_1= "AudioFiles\ozymandias_1_1.wav"#Sample 1, phrase 1
OZYMANDIAS1_2= "AudioFiles\ozymandias_1_2.wav"#Sample 1, phrase 2
OZYMANDIAS1_3= "AudioFiles\ozymandias_1_3.wav"#Sample 1, phrase 3
OZYMANDIAS2_1= "AudioFiles\ozymandias_2_1.wav"#Sample 2, phrase 1
OZYMANDIAS2_2= "AudioFiles\ozymandias_2_2.wav"#Sample 2, phrase 2
OZYMANDIAS2_3= "AudioFiles\ozymandias_2_3.wav"#Sample 2, phrase 3
OZYMANDIAS3_1= "AudioFiles\Ozymandias_3_1_fixed.wav"#Sample 3, phrase 1
OZYMANDIAS3_2= "AudioFiles\Ozymandias_3_2_fixed.wav"#Sample 3, phrase 2
OZYMANDIAS3_3= "AudioFiles\Ozymandias_3_3_fixed.wav"#Sample 3, phrase 3
#RHYME1="Rhyme1.wav" #Similar phrase, diff speaker
#RHYME2="Rhyme2.wav" #Similar phrase, diff speaker

In [57]:
model = whisper.load_model("base") #Try gpu

In [59]:
#MFCC pre_emphasis step
def pre_emphasis(signal, pre_emphasis_coefficient=0.97):
    return np.append(signal[0], signal[1:] - pre_emphasis_coefficient * signal[:-1])

In [60]:
#MFCC window framing step
def framing(signal, frame_size, frame_stride, sample_rate):
    frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate
    signal_length = len(signal)
    frame_length = int(round(frame_length))
    frame_step = int(round(frame_step))
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(signal, z)
    
    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    return frames

In [61]:
#MFCC hamming window step
def windowing(frames):
    return frames * np.hamming(frames.shape[1])

In [62]:
#MFCC power spectrum step
def power_spectrum(frames, NFFT):
    mag_frames = np.absolute(np.fft.rfft(frames, NFFT))  # Magnitude of the FFT
    pow_frames = ((1.0 / NFFT) * (mag_frames ** 2))  # Power Spectrum
    return pow_frames

In [63]:
#MFCC convert to mel scale step
def mel_filter_bank(pow_frames, nfilt, NFFT, sample_rate):
    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))  # Convert Hz to Mel
    mel_points = np.linspace(low_freq_mel, high_freq_mel, nfilt + 2)  # Equally spaced in Mel scale
    hz_points = (700 * (10**(mel_points / 2595) - 1))  # Convert Mel to Hz
    bin = np.floor((NFFT + 1) * hz_points / sample_rate)

    fbank = np.zeros((nfilt, int(np.floor(NFFT / 2 + 1))))
    for m in range(1, nfilt + 1):
        f_m_minus = int(bin[m - 1])   # left
        f_m = int(bin[m])             # center
        f_m_plus = int(bin[m + 1])    # right

        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])

    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)  # Numerical Stability
    filter_banks = 20 * np.log10(filter_banks)  # dB
    return filter_banks

In [64]:
#MFCC Discrete Cosine Transform step

def compute_mfcc(filter_banks, num_ceps):
    mfcc = dct(filter_banks, type=2, axis=1, norm='ortho')[:, 1 : (num_ceps + 1)]
    return mfcc

In [65]:
#MFCC aplitude smoother
def amplitude_envelope(signal, frame_size, hop_length):
    """Calculate the amplitude envelope of a signal with a given frame size nad hop length."""
    amplitude_envelope = []

    # calculate amplitude envelope for each frame
    for i in range(0, len(signal), hop_length):
        amplitude_envelope_current_frame = max(signal[i:i+frame_size])
        amplitude_envelope.append(amplitude_envelope_current_frame)


    return np.array(amplitude_envelope)

In [66]:
def fast_fourier_transform_denoise(values, length, timestep):
    # Compute the FFT of the input values
    fft_result = np.fft.fft(values)
    frequencies = np.fft.fftfreq(length, d=timestep)
    
    # Determine the cutoff frequency as the 25th percentile of the frequencies
    cutoff_frequency = np.percentile(abs(frequencies), 5)
    
    # Set the FFT results to zero for frequencies below the cutoff
    fft_result[abs(frequencies) < cutoff_frequency] = 0
    
    # Perform the inverse FFT to get the denoised signal
    denoised_signal = np.fft.ifft(fft_result)
    
    return np.real(denoised_signal)

In [67]:
#Get Derivative Helper(using slope approximation)
def get_derivative_of_list(data, time_step):
    #print(data)
    #print(time_step)    
    der_list=[]
    if(len(data)==0): 
        return der_list
    elif len(data)==1:
        der_list.append(0)
        return der_list
    der_list.append((data[1]-data[0])/time_step)
    for i in range(1,len(data)-1):
        der_list.append((data[i+1]-data[i-1])/(time_step*2)) #Takes the slope of secant line of adjacent two points
    der_list.append(data[len(data)-1]-data[len(data)-2]/(time_step))
    return der_list

In [68]:
#Normalizes a list helper
def get_normalized_list(data, ignoreZeros=False):
    normalization_factor=max(abs(min(data)),max(data))
    if normalization_factor == 0:
        raise ValueError("Average is zero, normalization cannot be performed")
        return [0 for x in data], 0
    return np.array(data)/normalization_factor, normalization_factor
    #"""data2=[abs(x) for x in data]
    #avg=np.nanmean(data2) if ( not ignoreZeros) else np.nanmean([x if (x>0) else np.nan for x in data2])
    #if avg == 0:
    #    raise ValueError("Average is zero, normalization cannot be performed")
    #    return [0 for x in data], 0
    #return np.array(data)/avg, avg"""

In [69]:
def process_audio_data(filename):
    MIN_SENTENCE_CONFIDENCE_THRESHOLD=0.4 #For sentence confidence. If model confidence is below the value, sentence is ignored.
    MIN_WORD_CONFIDENCE_THRESHOLD=0.3 #For word confidence. If model confidence is below the value, individual word is ignored.
    

    #TRANSCRIPTION PROCESS
    audio=whisper.load_audio(filename)
    results=whisper.transcribe(model, audio, language="en")
    #print(results)
    #AMPLITUDE CALCULATION
    FRAME_SIZE = 128
    HOP_LENGTH=128
    SAMPLE_RATE=16000.0
    MIN_PITCH=50
    MAX_PITCH=350   

    amplitude_values = amplitude_envelope(audio, FRAME_SIZE, HOP_LENGTH)
    frames = range(len(amplitude_values))
    t = librosa.frames_to_time(frames,sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
    amplitude_values=fast_fourier_transform_denoise(amplitude_values,len(amplitude_values),HOP_LENGTH/SAMPLE_RATE)

    #"""#tck = interpolate.splrep(t, amplitude_values, s=0)
    #xnew = np.linspace(0, t[-1], num=len(amplitude_values))
    #amplitude_values = interpolate.splev(xnew, tck, der=0)

    #pitch = pitch[np.nonzero(pitch)]  # Remove zero entries"""
    
    snd = parselmouth.Sound(filename)
    pitch = snd.to_pitch(time_step=(HOP_LENGTH/SAMPLE_RATE), pitch_floor=MIN_PITCH, pitch_ceiling=MAX_PITCH)
    pitch_values = pitch.selected_array['frequency']
    pitch_values=fast_fourier_transform_denoise(pitch_values,len(pitch_values),HOP_LENGTH/SAMPLE_RATE)

    #"""#tckp = interpolate.splrep(pitch.xs(), pitch_values, s=0)
    #newp = np.linspace(0, t[-1], num=len(pitch_values))
    #pitch_values = interpolate.splev(xnewp, tckp, der=0)"""


    #Account for rounding error causing different lengths
    if(len(pitch_values)>len(amplitude_values)):
        pitch_values=pitch_values[0:len(amplitude_values)]
        t=t[0:len(amplitude_values)]
    if(len(pitch_values)<len(amplitude_values)):
        amplitude_values=amplitude_values[0:len(pitch_values)]
        t=t[0:len(pitch_values)]
    pitch_values= [p if p>0 else 0 for p in pitch_values]
    
    amplitude_values, avg_amp=get_normalized_list(amplitude_values)
    pitch_values, avg_pitch=get_normalized_list(pitch_values,ignoreZeros=True)
    #Transcription Values Calculation
    lst = []

    for i in range(len(results["segments"])):
        if results["segments"][i]["confidence"]<MIN_SENTENCE_CONFIDENCE_THRESHOLD:
            print("Skipping Sentence "+str(i))
            continue #Skips sentence fully
        for j in range(len(results["segments"][i]["words"])):
            if results["segments"][i]["words"][j]["confidence"]<MIN_WORD_CONFIDENCE_THRESHOLD:
                print("Skipping Word "+str(i)+ ", "+str(j))
                continue #Skips word 
            lst.append([results["segments"][i]["words"][j]["text"],results["segments"][i]["words"][j]["start"],results["segments"][i]["words"][j]["end"],0,0,0,0,0,0,0]) #zero as placeholder for other values
            
    df_word = pd.DataFrame(lst, columns=['Word', 'Start', 'End','length','time_spent','std_time_spent','speed','post_space','amplitude','pitch'])
    
    #Data Calculation
    df_word["length"] = df_word["Word"].apply(len)
    df_word['time_spent'] = df_word['End'] - df_word['Start']
    df_word['std_time_spent'] = df_word['time_spent']/df_word['length']
    df_word["speed"] = df_word["length"]/df_word["time_spent"]
    

    post =  df_word['Start'][1:].values - df_word['End'][:-1].values
    df_word["time"] = df_word["Start"] + (df_word["End"] - df_word["Start"])/2

    df_word['post_space'] = np.append(post,[0])
    
    inte = []
    pit = []

    for i in range(len(df_word)):
            t1 = df_word['Start'][i]
            t2 = df_word['End'][i]
            idx = (t>t1) & (t<t2)

            int_avg = np.nanmean(amplitude_values[idx])
            inte.append(int_avg)
            
            pit_avg = np.nanmean(pitch_values[idx])
            pit.append(pit_avg)


    df_word['amplitude'] = inte
    df_word['pitch'] = pit
        
        
    return df_word, avg_amp, avg_pitch, t, amplitude_values,pitch_values

In [70]:
audioSample1=SONNET2
audioSample2=SONNET1
audioSamples=[TEDTALKBASE,TEDTALKNOISE,TEDTALKDELAYED,TEDTALKBASE2,TEDTALKNOISE2,TEDTALKDELAYED2,SONNET1,SONNET2,OZYMANDIAS1_1,OZYMANDIAS1_2,OZYMANDIAS1_3,OZYMANDIAS2_1,OZYMANDIAS2_2,OZYMANDIAS2_3,OZYMANDIAS3_1,OZYMANDIAS3_2,OZYMANDIAS3_3]

In [71]:
from audio_similarity import AudioSimilarity

# Paths to the original and compariosn audio files/folders

#original_path = audioSamples[0]
#compare_path = audioSamples[1]

# Set the sample rate and weights for the metrics
def audio_similarity(original_path,compare_path):
    sample_rate = 16000
    weights = {
        'zcr_similarity': 0.2,
        'rhythm_similarity': 0.2,
        'chroma_similarity': 0.2,
        'energy_envelope_similarity': 0.1,
        'spectral_contrast_similarity': 0.1,
        'perceptual_similarity': 0.2
    }

    verbose = True # Show logs

    # Create an instance of the AudioSimilarity class

    audio_similarity = AudioSimilarity(original_path, compare_path, sample_rate, weights, verbose=verbose)

    # Calculate a single metric

    #zcr_similarity = audio_similarity.zcr_similarity()

    # Calculate the Stent Weighted Audio Similarity
    #metrics='all'
    similarity_score = audio_similarity.stent_weighted_audio_similarity(metrics='swass') # You can select all metrics or just the 'swass' metric
    return similarity_score
"""
    print(f"Stent Weighted Audio Similarity: {similarity_score}")

    audio_similarity.plot(metrics=None,
                        option='radar',
                        figsize=(10, 7),
                        color1='red',
                        color2='green',
                        dpi=100,
                        savefig=False,
                        fontsize=6,
                        label_fontsize=8,
                        title_fontsize=14, 
                        alpha=0.5, 
                        title='Audio Similarity Metrics')"""

'\n    print(f"Stent Weighted Audio Similarity: {similarity_score}")\n\n    audio_similarity.plot(metrics=None,\n                        option=\'radar\',\n                        figsize=(10, 7),\n                        color1=\'red\',\n                        color2=\'green\',\n                        dpi=100,\n                        savefig=False,\n                        fontsize=6,\n                        label_fontsize=8,\n                        title_fontsize=14, \n                        alpha=0.5, \n                        title=\'Audio Similarity Metrics\')'

In [72]:
COEFFICIENT_NUM=5

In [185]:
import numpy as np
import scipy.io.wavfile as wav
from scipy.fftpack import dct
#from python_speech_features import mfcc
def calculate_mfcc(filename):
    # Load the audio file
    sample_rate, signal = wav.read(filename)
    signal = signal[0:int(3.5 * sample_rate)]  # Keep the first 3.5 seconds

    #signal=fast_fourier_transform_denoise(signal,len(signal),1/sample_rate)
    # Pre-emphasis
    emphasized_signal = pre_emphasis(signal)

    # Framing
    frame_size = 0.025
    frame_stride = 0.01
    frames = framing(emphasized_signal, frame_size, frame_stride, sample_rate)

    # Windowing
    frames = windowing(frames)

    # Fourier Transform and Power Spectrum
    NFFT = 512
    pow_frames = power_spectrum(frames, NFFT)

    #Maybe only look at the peaks
    # Mel Filter Bank
    nfilt = 40
    filter_banks = mel_filter_bank(pow_frames, nfilt, NFFT, sample_rate)

    # MFCCs
    mfcc=compute_mfcc(filter_banks,COEFFICIENT_NUM)
    #plot_mfcc_coefficients(mfcc)
    return mfcc

In [169]:
#MFCC data writer
"""import csv


# Specify the file path to save the CSV
file_path = 'MFCC2.csv'

# Write the 2D array data to a CSV file
with open(file_path, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(mfcc2)

print(f'Data has been written to {file_path}')"""

"import csv\n\n\n# Specify the file path to save the CSV\nfile_path = 'MFCC2.csv'\n\n# Write the 2D array data to a CSV file\nwith open(file_path, 'w', newline='') as file:\n    writer = csv.writer(file)\n    writer.writerows(mfcc2)\n\nprint(f'Data has been written to {file_path}')"

In [170]:
#MFCC specific distance
"""from dtaidistance import dtw
from dtaidistance import dtw_visualisation as dtwvis
from dtaidistance import dtw_ndim
import pandas as pd

# Assuming 'audioSamples' is defined
n = len(audioSamples)
distances = [[0] * n for _ in range(n)]  # Initialize a 2D array with zeros

for i in range(n):
    for j in range(i + 1, n):  # Start from i+1 to avoid redundant calculations
        distance = dtw_ndim.distance_fast(calculate_mfcc(audioSamples[i]),calculate_mfcc(audioSamples[j]))
        distances[i][j] = distance  # Set the distance in the upper triangle
        distances[j][i] = distance  # Mirror it in the lower triangle

# Convert the 2D list to a DataFrame
distances_df = pd.DataFrame(distances)

# Save the DataFrame to a CSV file
distances_df.to_csv('DistancesMFCC.csv', index=False, header=False)  # No index or headers

print("CSV file has been created and saved as 'distances.csv'")"""

'from dtaidistance import dtw\nfrom dtaidistance import dtw_visualisation as dtwvis\nfrom dtaidistance import dtw_ndim\nimport pandas as pd\n\n# Assuming \'audioSamples\' is defined\nn = len(audioSamples)\ndistances = [[0] * n for _ in range(n)]  # Initialize a 2D array with zeros\n\nfor i in range(n):\n    for j in range(i + 1, n):  # Start from i+1 to avoid redundant calculations\n        distance = dtw_ndim.distance_fast(calculate_mfcc(audioSamples[i]),calculate_mfcc(audioSamples[j]))\n        distances[i][j] = distance  # Set the distance in the upper triangle\n        distances[j][i] = distance  # Mirror it in the lower triangle\n\n# Convert the 2D list to a DataFrame\ndistances_df = pd.DataFrame(distances)\n\n# Save the DataFrame to a CSV file\ndistances_df.to_csv(\'DistancesMFCC.csv\', index=False, header=False)  # No index or headers\n\nprint("CSV file has been created and saved as \'distances.csv\'")'

In [171]:
#MFCC spectrum grapher
import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
def plot_mfcc(mfcc):
    fig, axs = plt.subplots(2, 1, figsize=(12, 8))

    # Plot MFCC for audio file 1
    im1 = axs[0].imshow(mfcc.T, aspect='auto', origin='lower', cmap='viridis')
    axs[0].set_title('MFCC - Audio File 1')
    axs[0].set_xlabel('Frame')
    axs[0].set_ylabel('MFCC Coefficient')
    plt.colorbar(im1, ax=axs[0], orientation='horizontal')

    plt.tight_layout()
    plt.show()

In [172]:
#MFCC Coefficinet Grapher
num_coeffs_to_plot = 5
def plot_mfcc_coefficients(mfcc):
    plt.figure(figsize=(10, 8))
    for i in range(num_coeffs_to_plot):
        plt.subplot(num_coeffs_to_plot, 1, i + 1)
        plt.plot(mfcc[:, i])#, label=f'Audio 1 - Coeff {i+1}')
        #plt.plot(mfcc2[:, i], label=f'Audio 2 - Coeff {i+1}', linestyle='--')
        #plt.plot(fast_fourier_transform_denoise(mfcc1[:, i],len(mfcc1[:, i]),1/sample_rate), label=f'Audio 1 - Coeff {i+1}')
        #plt.plot(fast_fourier_transform_denoise(mfcc2[:, i],len(mfcc2[:, i]),1/sample_rate), label=f'Audio 2 - Coeff {i+1}', linestyle='--')
        plt.title(f'MFCC Coefficient {i+1}')
        plt.xlabel('Frame')
        plt.ylabel('Amplitude')
        plt.legend()
    plt.tight_layout()
    plt.show()

In [173]:
#Euclidean Distance Graph
"""from scipy.spatial.distance import euclidean
distances = np.array([[euclidean(mfcc1[i], mfcc2[j]) for j in range(mfcc2.shape[0])] for i in range(mfcc1.shape[0])])

# Plot distance heatmap
plt.figure(figsize=(12, 8))
plt.imshow(distances, aspect='auto', origin='lower', cmap='hot')
plt.title('MFCC Frame-to-Frame Distance')
plt.xlabel('Frame (Audio 2)')
plt.ylabel('Frame (Audio 1)')
plt.colorbar(label='Euclidean Distance')
plt.show()"""

"from scipy.spatial.distance import euclidean\ndistances = np.array([[euclidean(mfcc1[i], mfcc2[j]) for j in range(mfcc2.shape[0])] for i in range(mfcc1.shape[0])])\n\n# Plot distance heatmap\nplt.figure(figsize=(12, 8))\nplt.imshow(distances, aspect='auto', origin='lower', cmap='hot')\nplt.title('MFCC Frame-to-Frame Distance')\nplt.xlabel('Frame (Audio 2)')\nplt.ylabel('Frame (Audio 1)')\nplt.colorbar(label='Euclidean Distance')\nplt.show()"

In [174]:
AMPLITUDE_METRIC_INDEX=0
PITCH_METRIC_INDEX=1
DAMPLITUDE_METRIC_INDEX=2
DPITCH_METRIC_INDEX=3

# metrics=['amplitude', 'pitch',  'dAmplitude','dPitch']
TIMESTEP_METRICS=[False, False, True, False]

In [175]:

def get_metric_distance(time1, amplitude1, pitch1, time2, amplitude2, pitch2, useDTW=True):
    lst1=[]
    lst2=[]
    #print(amplitude1)
    amplitude1 , _=get_normalized_list(amplitude1)
    amplitude2, _=get_normalized_list(amplitude2)
    pitch1, _=get_normalized_list(pitch1,ignoreZeros=True)
    pitch2, _=get_normalized_list(pitch2,ignoreZeros=True)
    
    amplitude1=amplitude1.real
    amplitude2=amplitude2.real
    pitch1=pitch1.real
    pitch2=pitch2.real

    
    if(TIMESTEP_METRICS[AMPLITUDE_METRIC_INDEX]):
        lst1.append(amplitude1.real)
        lst2.append(amplitude2.real)

    if(TIMESTEP_METRICS[PITCH_METRIC_INDEX]):
        lst1.append(pitch1.real)
        lst2.append(pitch2.real)

    if(TIMESTEP_METRICS[DAMPLITUDE_METRIC_INDEX]):
        dAmplitude1, _ = get_normalized_list(get_derivative_of_list(amplitude1.real,time1[1]-time1[0]))
        dAmplitude2, _ = get_normalized_list(get_derivative_of_list(amplitude2.real,time2[1]-time2[0]))
        lst1.append(np.array(dAmplitude1.real,dtype=np.double))
        lst2.append(np.array(dAmplitude2.real,dtype=np.double))

    if(TIMESTEP_METRICS[DPITCH_METRIC_INDEX]):
        dPitch1, _ = get_normalized_list(get_derivative_of_list(pitch1.real,time1[1]-time1[0]),ignoreZeros=True)
        dPitch2, _ = get_normalized_list(get_derivative_of_list(pitch2.real,time2[1]-time2[0]),ignoreZeros=True)
        lst1.append(np.array(dPitch1.real,dtype=np.double))
        lst2.append(np.array(dPitch2.real,dtype=np.double))
    
    lst1=np.array(lst1,dtype=np.double).T
    lst2=np.array(lst2,dtype=np.double).T
    print(lst1.shape)
    #plot_data(lst1,time1,lst2)
    if useDTW:
        return get_dtw_distance(lst1,lst2,True)
    return get_euclidean_distance(lst1,lst2)
    #d = get_dtw_distance(lst1,lst2,True)
    #print("D: ",d," t: ",len(time1))
    #return d

In [176]:
# columns=['Word', 'Start', 'End','length','time_spent','std_time_spent','speed','post_space','amplitude','pitch']
DFWORD_METRICS=['length','speed','amplitude','pitch']

In [177]:
def get_df_word_distance(df_word1,df_word2,useDTW=True):
    if useDTW:
        return get_dtw_distance(np.array(df_word1[DFWORD_METRICS],dtype=np.double),np.array(df_word2[DFWORD_METRICS],dtype=np.double),True)
    return get_euclidean_distance(np.array(df_word1[DFWORD_METRICS],dtype=np.double),np.array(df_word2[DFWORD_METRICS],dtype=np.double))

In [178]:
def get_mfcc_distance(i,j,useDTW=True):
    list1=calculate_mfcc(audioSamples[i])
    list2=calculate_mfcc(audioSamples[j])
    if useDTW: 
        return get_dtw_distance(list1,list2,True)/(math.sqrt(len(list1)*len(list2))*math.sqrt(COEFFICIENT_NUM))
    return get_euclidean_distance(list1,list2)/(math.sqrt(len(list1)*len(list2))*math.sqrt(COEFFICIENT_NUM))

In [179]:

"""
df_word1, _ , _, time1,amplitude1, pitch1=process_audio_data(audioSample1)
df_word2, _ , _, time2,amplitude2,pitch2 =process_audio_data(audioSample2) 

dAmplitude1=get_derivative_of_list(amplitude1,time1[1]-time1[0])
dPitch1=get_derivative_of_list(pitch1,time1[1]-time1[0])


dAmplitude2=get_derivative_of_list(amplitude2,time2[1]-time2[0])
dPitch2=get_derivative_of_list(pitch2,time2[1]-time2[0])

lst1=[amplitude1.astype(np.double)]#,np.array(dPitch1,dtype=np.double)]
lst2=[amplitude2.astype(np.double)]#,np.array(dPitch2,dtype=np.double)]
#lst1.append()
#lst2.append()
lst1=np.array(lst1,dtype=np.double).T
lst2=np.array(lst2,dtype=np.double).T"""
def get_total_distance(index1,index2,useDTW=True):

  df_word1, _ , _, time1,amplitude1, pitch1 =process_audio_data(audioSamples[index1])
  
  df_word2, _ , _, time2,amplitude2, pitch2 =process_audio_data(audioSamples[index2])
  print(math.sqrt(len(time1)*len(time2)))
  
  metric_distance = 20*get_metric_distance(time1,amplitude1,pitch1,time2,amplitude2,pitch2,useDTW)/(math.sqrt(len(time1)*len(time2))*math.sqrt(sum(TIMESTEP_METRICS)))
  #print("d1: ",df_word1)
  #print("d2: ",df_word2)
  dfword_distance = get_df_word_distance(df_word1,df_word2,useDTW)/(math.sqrt(len(df_word1)*len(df_word2))*math.sqrt(len(DFWORD_METRICS)))
  mfcc_distance = get_mfcc_distance(index1,index2,useDTW)

  METRIC_WEIGHT=1
  DFWORD_WEIGHT=1
  MFCC_WEIGHT=1

  d=metric_distance*METRIC_WEIGHT+dfword_distance*DFWORD_WEIGHT+mfcc_distance*MFCC_WEIGHT
  #"""
  #lst1,lst2= interpolate_and_match_length(lst1,lst2)

  #print(f"Size of lst1: {lst1.shape}")
  #print(f"Size of lst2: {lst2.shape}")
  #print(str(type()))#+str(amplitude2))"""


  return d, metric_distance, dfword_distance, mfcc_distance

In [180]:
def get_euclidean_distance(list1, list2):
    list1=np.array(list1,dtype=np.double)
    list2=np.array(list2,dtype=np.double)
    # Pad the shorter array if necessary
    if len(list1) > len(list2):
        if list2.ndim==1:
            list2 = np.pad(list2, (0, len(list1) - len(list2)), 'constant')
        else:
            list2 = np.pad(list2, ((0, len(list1) - len(list2)),(0,0)), 'constant')
    elif len(list2) > len(list1):
        if list1.ndim==1:
            list1 = np.pad(list1, (0, len(list2) - len(list1)), 'constant')
        else:
            list1 = np.pad(list1, ((0, len(list2) - len(list1)),(0,0)), 'constant')

    # Calculate Euclidean distance
    distance = np.linalg.norm(list1 - list2)
    return distance

In [181]:
def get_dtw_distance(list1, list2,multidimensional=False):
    
    if multidimensional:
        return dtw_ndim.distance_fast(list1,list2)
    else:
        return dtw.distance_fast(np.array(list1,dtype=np.double), np.array(list2,dtype=np.double))

In [182]:
def get_distance_from_index(index1, index2, useDTW=False):
    FRAME_SIZE = 128
    HOP_LENGTH=128
    
    audio1=whisper.load_audio(audioSamples[index1])
    amplitude_values1 = amplitude_envelope(audio1, FRAME_SIZE, HOP_LENGTH)
    
    audio2=whisper.load_audio(audioSamples[index2])
    amplitude_values2 = amplitude_envelope(audio2, FRAME_SIZE, HOP_LENGTH)
    #plot_data(amplitude_values1,range(math.ceil(len(amplitude_values1)/FRAME_SIZE)),amplitude_values2)

    return get_dtw_distance(amplitude_values1,amplitude_values2,False) if useDTW else get_euclidean_distance(amplitude_values1,amplitude_values2)

In [183]:
#audioSamples=[TEDTALKBASE,TEDTALKNOISE,TEDTALKDELAYED,TEDTALKBASE2,TEDTALKNOISE2,TEDTALKDELAYED2,SONNET1,SONNET2,OZYMANDIAS1_1,OZYMANDIAS1_2,OZYMANDIAS1_3,OZYMANDIAS2_1,OZYMANDIAS2_2,OZYMANDIAS2_3,OZYMANDIAS3_1,OZYMANDIAS3_2,OZYMANDIAS3_3]
COMPARISON_INDEXES=[[0,1],[0,2],[1,2],[3,4],[3,5],[4,5], #Ted Talks (Same Clip, Clip w/noise, Clip w/delay) #Index 0-5
                    [6,7],  #Sonnets (Same Speaker, Same Content) #Index 6
                    [8,11],[8,14],[11,14],[9,12],[9,15],[12,15],[10,13],[10,16],[13,16], #Ozymandias Pairs (Same content) #Index 7-15
                    [8,9],[8,10],[9,10],[11,12],[11,13],[12,13],[14,15],[14,16],[15,16], #Ozymandias (Same Spaker) #Index 16-24
                    [0,3], #Ted Talk(Same Speaker) #Index 25
                    #Different content, Different Speaker
                    [0,6],[0,7],[0,8],[0,9],[0,10],[0,11],[0,12],[0,13],[0,14],[0,15],[0,16], #Index 26-36
                    [3,6],[3,7],[3,8],[3,9],[3,10],[3,11],[3,12],[3,13],[3,14],[3,15],[3,16], #Index 37-47
                    [6,8],[6,9],[6,10],[6,11],[6,12],[6,13],[6,14],[6,15],[6,16], #Index 48-56
                    [7,8],[7,9],[7,10],[7,11],[7,12],[7,13],[7,14],[7,15],[7,16]  #Index 57-65
                    ]


#[[0,1],[0,2],[1,2],[3,4],[5,8],[5,11],[8,11],[6,9],[6,12],[9,12],[7,10],[7,13],[10,13],[0,3],[0,5],[0,9],[0,13],[1,4],[1,4],[1,8],[1,12],[1,7],[2,3],[2,11],[2,6],[2,10],[3,5],[3,9],[3,13],[4,8],[4,12],[4,7]]

In [ ]:
n = len(COMPARISON_INDEXES)

USEDTW=False
distances = [[],[],[],[],[],[],[]]# for _ in range(n)]  # Initialize a 2D array with zeros

#Calculates 2D grid of distances where [i,j] == [j,i] and [i,i]==0
for i in range(len(COMPARISON_INDEXES)):
    distance, metric_distance, dfword_distance, mfcc_distance = get_total_distance(COMPARISON_INDEXES[i][0], COMPARISON_INDEXES[i][1],USEDTW)  # Calculate distance only once
    signal_distance=get_distance_from_index(COMPARISON_INDEXES[i][0], COMPARISON_INDEXES[i][1],USEDTW)
    similarity=audio_similarity(audioSamples[COMPARISON_INDEXES[i][0]],audioSamples[COMPARISON_INDEXES[i][1]])
    print(distance, " ", metric_distance, " ",dfword_distance, " ",mfcc_distance)
    #2D array assignment
    distances[0].append(distance)
    distances[1].append(metric_distance)
    distances[2].append(dfword_distance)
    distances[3].append(mfcc_distance)
    distances[4].append(signal_distance)
    distances[5].append(1-similarity)
# Convert the 2D list to a DataFrame
distances_df = pd.DataFrame(distances)
distances_df=distances_df.T

# Save the DataFrame to a CSV file
distances_df.to_csv('DistancesEuclidean.csv', index=False, header=False)  # No index or headers

print("CSV file has been created and saved as 'distances.csv'")

100%|██████████| 800/800 [00:01<00:00, 488.59frames/s]


Skipping Word 0, 0
993.0
(993, 1)


Loading comparison files:: 100%|██████████| 1/1 [00:00<00:00, 100.01it/s]
Calculating zero crossing rate similarity...
Calculating rhythm similarity...
Calculating chroma similarity similarity...
Calculating energy envelope similarity...
Calculating spectral contrast similarity...
Calculating perceptual similarity...


5.209547258352095   0.02051900901765391   1.571145512139447   3.6178827371949938


100%|██████████| 900/900 [00:01<00:00, 570.54frames/s]


...

Skipping Word 0, 5


100%|██████████| 600/600 [00:01<00:00, 462.99frames/s]


429.54860027708156
(248, 1)


Loading comparison files:: 100%|██████████| 1/1 [00:00<00:00, 52.36it/s]
Calculating zero crossing rate similarity...
Calculating rhythm similarity...
Calculating chroma similarity similarity...
Calculating energy envelope similarity...
Calculating spectral contrast similarity...
Calculating perceptual similarity...


4.981804082777316   0.25967436810450295   2.233803777793611   2.4883259368792023
CSV file has been created and saved as 'distances.csv'


In [188]:
"""n = len(COMPARISON_INDEXES)

USEDTW=True
metric_distances = [[] for _ in range(2**len(TIMESTEP_METRICS))]  # Initialize a 2D array with zeros

#Calculates 2D grid of distances where [i,j] == [j,i] and [i,i]==0
for j in range(1,2**len(TIMESTEP_METRICS)):
    TIMESTEP_METRICS= [(j >> k) % 2 == 1 for k in range(len(TIMESTEP_METRICS))]
    print(TIMESTEP_METRICS)
    for i in range(len(COMPARISON_INDEXES)):
        distance, metric_distance, dfword_distance, mfcc_distance = get_total_distance(COMPARISON_INDEXES[i][0], COMPARISON_INDEXES[i][1],USEDTW)  # Calculate distance only once
        print(metric_distance)
        metric_distances[j].append(metric_distance)
# Convert the 2D list to a DataFrame
metric_distances_df = pd.DataFrame(metric_distances)
metric_distances_df=metric_distances_df.T

# Save the DataFrame to a CSV file
metric_distances_df.to_csv('DistancesMetricOnly.csv', index=False, header=False)  # No index or headers

print("CSV file has been created and saved as 'distances.csv'")"""

'n = len(COMPARISON_INDEXES)\n\nUSEDTW=True\nmetric_distances = [[] for _ in range(2**len(TIMESTEP_METRICS))]  # Initialize a 2D array with zeros\n\n#Calculates 2D grid of distances where [i,j] == [j,i] and [i,i]==0\nfor j in range(1,2**len(TIMESTEP_METRICS)):\n    TIMESTEP_METRICS= [(j >> k) % 2 == 1 for k in range(len(TIMESTEP_METRICS))]\n    print(TIMESTEP_METRICS)\n    for i in range(len(COMPARISON_INDEXES)):\n        distance, metric_distance, dfword_distance, mfcc_distance = get_total_distance(COMPARISON_INDEXES[i][0], COMPARISON_INDEXES[i][1],USEDTW)  # Calculate distance only once\n        print(metric_distance)\n        metric_distances[j].append(metric_distance)\n# Convert the 2D list to a DataFrame\nmetric_distances_df = pd.DataFrame(metric_distances)\nmetric_distances_df=metric_distances_df.T\n\n# Save the DataFrame to a CSV file\nmetric_distances_df.to_csv(\'DistancesMetricOnly.csv\', index=False, header=False)  # No index or headers\n\nprint("CSV file has been created and

In [ ]:
metric_distances_df = pd.DataFrame(metric_distances)
metric_distances_df=metric_distances_df.T

# Save the DataFrame to a CSV file
metric_distances_df.to_csv('DistancesMetricOnly.csv', index=False, header=False)  # No index or headers

In [186]:
def plot_data(lst1,time1,lst2):
    #Metric grapher
    plt.figure(figsize=(8, 4.5))  # Create a new figure for each combination of segments
    plottedIndex=1

    max_x_range=time1[-1]
    dat = lst1#lst1[:,plottedIndex]

    time_sequence = np.linspace(0, max_x_range, num=len(dat))

    plt.plot(time_sequence, dat, label=f'Set 1')  # Adjust label as necessary

    dat2 = lst2#lst2[:,plottedIndex]
    time_sequence2 = np.linspace(0, max_x_range, num=len(dat2))

    plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary
    plt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found

    plt.legend()
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.title(f'Amplitude Comparison')
    plt.show()

In [ ]:
#Metric grapher small
"""plt.figure()  # Create a new figure for each combination of segments
plottedIndex=0

max_x_range=time1[-1]
dat = lst1[:,plottedIndex]

time_sequence = np.linspace(0, max_x_range, num=len(dat))

plt.plot(time_sequence, dat)#, label=f'Set 1')  # Adjust label as necessary

dat2 = lst2[:,plottedIndex]
time_sequence2 = np.linspace(0, max_x_range, num=len(dat2))
    
#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary
plt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found

ax = plt.gca()
ax.set_aspect(aspect=1/0.35) 

plt.legend()
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title(f'Phrase Amplitude')
plt.show()"""

"plt.figure()  # Create a new figure for each combination of segments\nplottedIndex=0\n\nmax_x_range=time1[-1]\ndat = lst1[:,plottedIndex]\n\ntime_sequence = np.linspace(0, max_x_range, num=len(dat))\n\nplt.plot(time_sequence, dat)#, label=f'Set 1')  # Adjust label as necessary\n\ndat2 = lst2[:,plottedIndex]\ntime_sequence2 = np.linspace(0, max_x_range, num=len(dat2))\n    \n#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary\nplt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found\n\nax = plt.gca()\nax.set_aspect(aspect=1/0.35) \n\nplt.legend()\nplt.xlabel('Time')\nplt.ylabel('Amplitude')\nplt.title(f'Phrase Amplitude')\nplt.show()"

In [ ]:
#Metric Grapher #3 Pitch1
"""y, sr = librosa.load(audioSamples[1])
pitches, magnitude=librosa.piptrack(y=y,sr=sr)
pitch = np.max(pitches, axis=0)
pitch_avg=np.nanmean(pitch)
pitch=[p/pitch_avg for p in pitch]
plt.figure()  # Create a new figure for each combination of segments
max_x_range=10
time_sequence = np.linspace(0, max_x_range, num=len(pitch))

plt.plot(time_sequence, pitch)#, label=f'Set 1')  # Adjust label as necessary

    
#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary
plt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found

ax = plt.gca()
ax.set_aspect(aspect=1/0.35) 

plt.legend()
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title(f'Phrase Amplitude')
plt.show()"""

"y, sr = librosa.load(audioSamples[1])\npitches, magnitude=librosa.piptrack(y=y,sr=sr)\npitch = np.max(pitches, axis=0)\npitch_avg=np.nanmean(pitch)\npitch=[p/pitch_avg for p in pitch]\nplt.figure()  # Create a new figure for each combination of segments\nmax_x_range=10\ntime_sequence = np.linspace(0, max_x_range, num=len(pitch))\n\nplt.plot(time_sequence, pitch)#, label=f'Set 1')  # Adjust label as necessary\n\n    \n#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary\nplt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found\n\nax = plt.gca()\nax.set_aspect(aspect=1/0.35) \n\nplt.legend()\nplt.xlabel('Time')\nplt.ylabel('Amplitude')\nplt.title(f'Phrase Amplitude')\nplt.show()"

In [ ]:
#Metric Grapher #4 Pitch2
"""FRAME_SIZE = 128
HOP_LENGTH=128
SAMPLE_RATE=16000.0  
snd = parselmouth.Sound(audioSamples[1])
pitch = snd.to_pitch(time_step=(HOP_LENGTH/SAMPLE_RATE), pitch_floor=50, pitch_ceiling=350)
pitch_values = pitch.selected_array['frequency']

pitch_avg=np.nanmean(pitch_values)
pitch_values=[p/pitch_avg for p in pitch_values]

plt.figure()  # Create a new figure for each combination of segments
max_x_range=10
time_sequence = np.linspace(0, max_x_range, num=len(pitch_values))

plt.plot(time_sequence, pitch_values)#, label=f'Set 1')  # Adjust label as necessary

    
#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary
plt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found

ax = plt.gca()
ax.set_aspect(aspect=1/0.35) 

plt.legend()
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title(f'Phrase Amplitude')
plt.show()"""

"FRAME_SIZE = 128\nHOP_LENGTH=128\nSAMPLE_RATE=16000.0  \nsnd = parselmouth.Sound(audioSamples[1])\npitch = snd.to_pitch(time_step=(HOP_LENGTH/SAMPLE_RATE), pitch_floor=50, pitch_ceiling=350)\npitch_values = pitch.selected_array['frequency']\n\npitch_avg=np.nanmean(pitch_values)\npitch_values=[p/pitch_avg for p in pitch_values]\n\nplt.figure()  # Create a new figure for each combination of segments\nmax_x_range=10\ntime_sequence = np.linspace(0, max_x_range, num=len(pitch_values))\n\nplt.plot(time_sequence, pitch_values)#, label=f'Set 1')  # Adjust label as necessary\n\n    \n#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary\nplt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found\n\nax = plt.gca()\nax.set_aspect(aspect=1/0.35) \n\nplt.legend()\nplt.xlabel('Time')\nplt.ylabel('Amplitude')\nplt.title(f'Phrase Amplitude')\nplt.show()"

In [ ]:
#Metric Grapher #5 Pitch3 Aubio
"""import aubio

# Open the audio file
filename = audioSamples[1]
samplerate = 44100  # use original samplerate
win_s = 1024  # fft size
hop_s = 128   # hop size

s = aubio.source(filename, int(samplerate), hop_s)
samplerate = s.samplerate

# Create a pitch detection object
tolerance = 0.8
pitch_o = aubio.pitch("default", win_s, hop_s, int(samplerate))
pitch_o.set_unit("Hz")
pitch_o.set_tolerance(tolerance)

pitches = []
confidences = []

# Processing loop
while True:
    samples, read = s()
    pitch = pitch_o(samples)[0]
    confidence = pitch_o.get_confidence()
    if confidence > 0.2:
        pitches.append(pitch)
        confidences.append(confidence)
    if read < hop_s:
        break

print("Estimated pitches:", pitches)


pitch_avg=np.nanmean(pitches)
pitches=[p/pitch_avg for p in pitches]

plt.figure()  # Create a new figure for each combination of segments
max_x_range=10
time_sequence = np.linspace(0, max_x_range, num=len(pitches))

plt.plot(time_sequence, pitches)#, label=f'Set 1')  # Adjust label as necessary

    
#plt.plot(time_sequence2, dat2, c='red', label=f'Set 2')  # Adjust label as necessary
plt.xlim(0, max_x_range)  # Set the x-axis limits to the maximum found

ax = plt.gca()
ax.set_aspect(aspect=1/0.35) 

plt.legend()
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title(f'Phrase Amplitude')
plt.show()"""

'import aubio\n\n# Open the audio file\nfilename = audioSamples[1]\nsamplerate = 44100  # use original samplerate\nwin_s = 1024  # fft size\nhop_s = 128   # hop size\n\ns = aubio.source(filename, int(samplerate), hop_s)\nsamplerate = s.samplerate\n\n# Create a pitch detection object\ntolerance = 0.8\npitch_o = aubio.pitch("default", win_s, hop_s, int(samplerate))\npitch_o.set_unit("Hz")\npitch_o.set_tolerance(tolerance)\n\npitches = []\nconfidences = []\n\n# Processing loop\nwhile True:\n    samples, read = s()\n    pitch = pitch_o(samples)[0]\n    confidence = pitch_o.get_confidence()\n    if confidence > 0.2:\n        pitches.append(pitch)\n        confidences.append(confidence)\n    if read < hop_s:\n        break\n\nprint("Estimated pitches:", pitches)\n\n\npitch_avg=np.nanmean(pitches)\npitches=[p/pitch_avg for p in pitches]\n\nplt.figure()  # Create a new figure for each combination of segments\nmax_x_range=10\ntime_sequence = np.linspace(0, max_x_range, num=len(pitches))\n\nplt.

In [ ]:
def plot_dtw_graph(lst1,lst2):
    d, paths = dtw.warping_paths(lst1, lst2, window=25, psi=2)
    best_path = dtw.best_path(paths)
    dtwvis.plot_warpingpaths(lst1, lst2, paths, best_path)